In [1]:
"""
1. From list of Test set, get all impainting outputs
    - Store file_path, file_name, dataset, is_fake, full_data_pred_label, full_data_pred_confident
2.
"""

'\n1. From list of Test set, get all impainting outputs\n    - Store file_path, file_name, dataset, is_fake, full_data_pred_label, full_data_pred_confident\n2. \n'

In [2]:
import pandas as pd
import numpy as np
import torch
import os
import torch.nn as nn
import torch.optim as optim
from PIL import Image
from tqdm.auto import tqdm
from collections import defaultdict
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms
from transformers import CLIPVisionModel

from google.colab import drive

In [3]:
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Setup

In [4]:
# Load Test Table
test_csv_path = '/content/drive/MyDrive/TrainingData/test_set.xlsx'
test_table = pd.read_excel(test_csv_path)
test_table = test_table.copy()
test_table = test_table[['file_path', 'file_name', 'dataset', 'is_fake', 'full_data_pred_label', 'full_data_pred_confident']]
test_table.head()

,file_path,file_name,dataset,is_fake,full_data_pred_label,full_data_pred_confident
0,/content/drive/MyDrive/TrainingData/imagenet_a...,878_sdv5_00027.png,imagenet_ai_0424_sdv5,1,1,0.9941
1,/content/drive/MyDrive/TrainingData/imagenet_a...,882_sdv5_00039.png,imagenet_ai_0424_sdv5,1,1,0.5430
2,/content/drive/MyDrive/TrainingData/imagenet_a...,903_sdv5_00009.png,imagenet_ai_0424_sdv5,1,1,0.9639
3,/content/drive/MyDrive/TrainingData/imagenet_a...,907_sdv5_00020.png,imagenet_ai_0424_sdv5,1,1,1.0000
4,/content/drive/MyDrive/TrainingData/imagenet_a...,918_sdv5_00039.png,imagenet_ai_0424_sdv5,1,1,1.0000


In [5]:
# Get inpainting Output
inpainting_directory = '/content/drive/MyDrive/output_impainting_sdxl'

test_table['inpainting_path'] = inpainting_directory + '/' + test_table['file_name'].str.split('.').str[0] + '_out_SDXL.png'
test_table.head()

,file_path,file_name,dataset,is_fake,full_data_pred_label,full_data_pred_confident,inpainting_path
0,/content/drive/MyDrive/TrainingData/imagenet_a...,878_sdv5_00027.png,imagenet_ai_0424_sdv5,1,1,0.9941,/content/drive/MyDrive/output_impainting_sdxl/...
1,/content/drive/MyDrive/TrainingData/imagenet_a...,882_sdv5_00039.png,imagenet_ai_0424_sdv5,1,1,0.5430,/content/drive/MyDrive/output_impainting_sdxl/...
2,/content/drive/MyDrive/TrainingData/imagenet_a...,903_sdv5_00009.png,imagenet_ai_0424_sdv5,1,1,0.9639,/content/drive/MyDrive/output_impainting_sdxl/...
3,/content/drive/MyDrive/TrainingData/imagenet_a...,907_sdv5_00020.png,imagenet_ai_0424_sdv5,1,1,1.0000,/content/drive/MyDrive/output_impainting_sdxl/...
4,/content/drive/MyDrive/TrainingData/imagenet_a...,918_sdv5_00039.png,imagenet_ai_0424_sdv5,1,1,1.0000,/content/drive/MyDrive/output_impainting_sdxl/...


In [6]:
# Load Discriminator
MODEL_PATH = "/content/drive/MyDrive/Model/clip_model"
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SAVE_DIR = "/content/drive/MyDrive/Model/clip_classification_v5"

class ClassificationCLIP(nn.Module):
    def __init__(self, model_path):
        super().__init__()
        self.vision_encoder = CLIPVisionModel.from_pretrained(model_path)
        hidden_size = self.vision_encoder.config.hidden_size
        self.classifier = nn.Linear(hidden_size, 1)

    def forward(self, pixel_values):
        outputs = self.vision_encoder(pixel_values=pixel_values)
        return self.classifier(outputs.pooler_output)

model = ClassificationCLIP(MODEL_PATH).to(DEVICE)
model.load_state_dict(torch.load(
    os.path.join(SAVE_DIR, 'best_model.pth'),
    map_location=DEVICE, weights_only=False
))
model.eval()

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

CLIPVisionModel LOAD REPORT from: /content/drive/MyDrive/Model/clip_model
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.layer_norm2.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm1.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.bias   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.weight | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.v_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.q

ClassificationCLIP(
  (vision_encoder): CLIPVisionModel(
    (vision_model): CLIPVisionTransformer(
      (embeddings): CLIPVisionEmbeddings(
        (patch_embedding): Conv2d(3, 1024, kernel_size=(14, 14), stride=(14, 14), bias=False)
        (position_embedding): Embedding(257, 1024)
      )
      (pre_layrnorm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
      (encoder): CLIPEncoder(
        (layers): ModuleList(
          (0-23): 24 x CLIPEncoderLayer(
            (self_attn): CLIPAttention(
              (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
              (v_proj): Linear(in_features=1024, out_features=1024, bias=True)
              (q_proj): Linear(in_features=1024, out_features=1024, bias=True)
              (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
            )
            (layer_norm1): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
            (mlp): CLIPMLP(
              (activation_fn): QuickGELUActiv

# Evaluation

In [7]:
BATCH_SIZE = 32

# Standard CLIP image transformations
clip_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.48145466, 0.4578275, 0.40821073],
                         std=[0.26862954, 0.26130258, 0.27577711])
])

In [8]:
# Dataset designed specifically to read from the DataFrame
class InpaintingEvalDataset(Dataset):
    def __init__(self, df, transform):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_path = self.df.loc[idx, 'inpainting_path']
        try:
            img = Image.open(img_path).convert('RGB')
            img = self.transform(img)
        except Exception as e:
            # Fallback to a zero tensor if the generated image is missing/corrupted
            img = torch.zeros(3, 224, 224)
        return img, idx

# Setup the DataLoader
eval_ds = InpaintingEvalDataset(test_table, clip_transform)
eval_dl = DataLoader(eval_ds, batch_size=BATCH_SIZE, shuffle=False,
                     num_workers=2, pin_memory=True)

In [9]:
# Inference Loop
all_preds = []
all_confs = []
all_indices = []

print("Starting evaluation on inpainted images...")
with torch.no_grad():
    for imgs, idxs in tqdm(eval_dl, desc="Evaluating Inpainted Images"):
        imgs = imgs.to(DEVICE)

        # Use autocast for faster inference if running on a CUDA GPU
        if DEVICE.type == 'cuda':
            with torch.autocast(device_type='cuda', dtype=torch.float16):
                logits = model(imgs)
        else:
            logits = model(imgs)

        # 1. Get the probability of the image being REAL
        real_probs = torch.sigmoid(logits).squeeze(-1)

        # Handle the edge case where batch size is 1 (e.g., very end of dataset)
        if real_probs.dim() == 0:
            real_probs = real_probs.unsqueeze(0)

        # 2. Invert it to get the probability of the image being FAKE
        fake_probs = 1.0 - real_probs

        # 3. Predict based on the FAKE probability
        # (If Fake probability is >= 0.5, predict 1 for Fake)
        preds = (fake_probs >= 0.5).int()

        # 4. Save the FAKE predictions and FAKE confidences
        all_preds.extend(preds.cpu().numpy())
        all_confs.extend(fake_probs.cpu().numpy())
        all_indices.extend(idxs.numpy())

Starting evaluation on inpainted images...


Evaluating Inpainted Images:   0%|          | 0/32 [00:00<?, ?it/s]

# Save Result

In [10]:
# Ensure indices map correctly
results_df = pd.DataFrame({
    'index': all_indices,
    'reevaluation_pred_label': all_preds,
    'reevaluation_pred_confident': all_confs
}).set_index('index')

# Assign the new columns to the original table
test_table['reevaluation_pred_label'] = results_df['reevaluation_pred_label']
test_table['reevaluation_pred_confident'] = results_df['reevaluation_pred_confident']

test_table.to_csv('/content/drive/MyDrive/TrainingData/reevaluation.csv')

print("Evaluation complete! Here are the first few rows:")
test_table.head()

Evaluation complete! Here are the first few rows:


/usr/local/lib/python3.12/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: overflow encountered in cast
  has_large_values = (abs_vals > 1e6).any()


,file_path,file_name,dataset,is_fake,full_data_pred_label,full_data_pred_confident,inpainting_path,reevaluation_pred_label,reevaluation_pred_confident
0,/content/drive/MyDrive/TrainingData/imagenet_a...,878_sdv5_00027.png,imagenet_ai_0424_sdv5,1,1,0.9941,/content/drive/MyDrive/output_impainting_sdxl/...,1,0.865723
1,/content/drive/MyDrive/TrainingData/imagenet_a...,882_sdv5_00039.png,imagenet_ai_0424_sdv5,1,1,0.5430,/content/drive/MyDrive/output_impainting_sdxl/...,0,0.331055
2,/content/drive/MyDrive/TrainingData/imagenet_a...,903_sdv5_00009.png,imagenet_ai_0424_sdv5,1,1,0.9639,/content/drive/MyDrive/output_impainting_sdxl/...,1,0.755859
3,/content/drive/MyDrive/TrainingData/imagenet_a...,907_sdv5_00020.png,imagenet_ai_0424_sdv5,1,1,1.0000,/content/drive/MyDrive/output_impainting_sdxl/...,1,0.544434
4,/content/drive/MyDrive/TrainingData/imagenet_a...,918_sdv5_00039.png,imagenet_ai_0424_sdv5,1,1,1.0000,/content/drive/MyDrive/output_impainting_sdxl/...,1,0.926270
